# Day 5 · Concept 5 of 5: The Fire Drill (Putting It Together)

**Big idea:** do not hope your agent survives failures. Make failures happen on purpose and watch.
*About 20 minutes. This is the notebook where everything from today works together.*

---

### Where we are
Today's pieces, one line each:

| Notebook | Piece | One line |
| --- | --- | --- |
| 1 | Timeout | Hang up after a while |
| 2 | Retry, backoff, jitter | Redial politely, a few times only |
| 3 | Fallback and escape path | Plan B for temporary trouble, human for permanent trouble |
| 4 | Rate limit and backpressure | Protect the tool from crowds |

### What went wrong in L2
Failures only showed up by accident, in front of customers. Nobody had practised them.

### What we do now (L3)
We put the pieces into **one SupportDesk graph**, then run **fire drills**: failures we trigger deliberately, so we can prove every run ends safely.

In [ ]:
# Step 0 - Connect to the AI model (Ollama). Run this cell first.
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gpt-oss:120b-cloud", temperature=0)
try:
    print("Model says:", llm.invoke("Reply with just the word: ready").content)
except Exception as problem:
    print("The model is not reachable yet.")
    print("Fix: open a terminal, run   ollama signin   then restart this notebook.")
    print("Technical detail:", type(problem).__name__)

### The SupportDesk reliability graph
```
START -> [ call_tool ] --ok-----------------> [ write_reply ] -> END
            ^   |  \--temporary, tries left--> [ backoff ] --+
            +---|---------------------------------------------+
                |  \--temporary, out of tries-> [ fallback ] -> END
                \--permanent------------------> [ escalate ] -> END
```
`call_tool` already contains the **turnstile** and the **timeout**. Every step also writes a short line into a **trace** (a trail of what happened, like Day 4).

In [ ]:
import time
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

import operator
import random
import uuid
from typing import Annotated
from concurrent.futures import ThreadPoolExecutor, TimeoutError as WaitedTooLong

# ---- the settings panel: change these and see what happens ----
MCP_TIMEOUT_S = 0.3        # hang-up rule
MCP_ATTEMPTS = 2           # tries (same value as Day 19)
BASE_WAIT = 0.1            # first backoff
RATE_LIMIT, RATE_WINDOW = 4, 1.0

# ---- the fire drills: what the pretend card company does on each call ----
FAULTS = {
    "healthy":   ["ok"],
    "flaky":     ["error", "ok"],
    "down":      ["error", "error"],
    "slow":      ["slow", "ok"],
    "bad_input": ["ok"],
}

pool = ThreadPoolExecutor(max_workers=4)
recent_calls = []

def pretend_card_company(user, fault):
    if not (user.startswith("cust-") and user[5:].isdigit()):
        raise ValueError("user id must look like cust-123")      # permanent
    if fault == "slow":
        time.sleep(1.0)                                          # longer than our timeout
    if fault == "error":
        raise ConnectionError("line dropped")                    # temporary
    return "1,250 INR"

class State(TypedDict):
    scenario: str
    user: str
    cid: str
    t0: float
    attempts: int
    error: str
    fatal: bool
    fallback: bool
    balance: str
    answer: str
    trace: Annotated[list, operator.add]

def tr(state, node, detail):
    return f"cid={state['cid']} {node} +{int((time.time() - state['t0']) * 1000)}ms {detail}"

def wait_turn():
    now = time.time()
    recent_calls[:] = [t for t in recent_calls if now - t < RATE_WINDOW]
    if len(recent_calls) >= RATE_LIMIT:
        time.sleep(RATE_WINDOW - (now - recent_calls[0]))
    recent_calls.append(time.time())

def call_tool(state):
    wait_turn()                                                  # 1. turnstile
    attempt = state["attempts"] + 1
    fault = FAULTS[state["scenario"]][attempt - 1]
    future = pool.submit(pretend_card_company, state["user"], fault)
    try:
        balance = future.result(timeout=MCP_TIMEOUT_S)           # 2. timeout
        return {"attempts": attempt, "error": "", "balance": balance,
                "trace": [tr(state, "call_tool", f"attempt={attempt} ok")]}
    except WaitedTooLong:
        return {"attempts": attempt, "error": "TimeoutError",
                "trace": [tr(state, "call_tool", f"attempt={attempt} TimeoutError")]}
    except ValueError as problem:                                # permanent
        return {"attempts": attempt, "error": str(problem), "fatal": True,
                "trace": [tr(state, "call_tool", f"attempt={attempt} {problem} (permanent)")]}
    except ConnectionError:                                      # temporary
        return {"attempts": attempt, "error": "ConnectionError",
                "trace": [tr(state, "call_tool", f"attempt={attempt} ConnectionError")]}

def after_call_tool(state):       # the whole policy in four lines
    if state["error"] == "":
        return "done"
    if state["fatal"]:
        return "escape"
    if state["attempts"] < MCP_ATTEMPTS:
        return "retry"
    return "give_up"

def backoff(state):
    pause = BASE_WAIT * 2 ** (state["attempts"] - 1) + random.uniform(0, 0.05)
    time.sleep(pause)
    return {"trace": [tr(state, "backoff", f"waited {pause:.2f}s")]}

def fallback(state):
    return {"fallback": True, "answer": "Service busy, please try again later.",
            "trace": [tr(state, "fallback", f"FALLBACK after {state['attempts']} attempts")]}

def escalate(state):
    return {"answer": "ESCALATED to a human queue (no automatic retry)",
            "trace": [tr(state, "escalate", "escape path")]}

def write_reply(state):
    text = llm.invoke("You are a polite bank assistant. Say this in one short sentence: the balance is " + state["balance"]).content
    return {"answer": text, "trace": [tr(state, "write_reply", "model wrote the reply")]}

builder = StateGraph(State)
for name, fn in [("call_tool", call_tool), ("backoff", backoff), ("fallback", fallback),
                 ("escalate", escalate), ("write_reply", write_reply)]:
    builder.add_node(name, fn)
builder.add_edge(START, "call_tool")
builder.add_conditional_edges("call_tool", after_call_tool,
    {"done": "write_reply", "retry": "backoff", "give_up": "fallback", "escape": "escalate"})
builder.add_edge("backoff", "call_tool")
builder.add_edge("write_reply", END)
builder.add_edge("fallback", END)
builder.add_edge("escalate", END)
supportdesk = builder.compile()

def run_scenario(scenario, user="cust-1"):
    recent_calls.clear()
    return supportdesk.invoke({"scenario": scenario, "user": user, "cid": uuid.uuid4().hex[:6], "t0": time.time(),
                               "attempts": 0, "error": "", "fatal": False, "fallback": False,
                               "balance": "", "answer": "", "trace": []},
                              config={"recursion_limit": 12})      # the seat belt from notebook 2

### Run the five fire drills
For each one, **say out loud what you expect**, then run. The trace shows exactly what happened.

In [ ]:
for scenario, user in [("healthy", "cust-1"), ("flaky", "cust-1"), ("down", "cust-1"), ("slow", "cust-1"), ("bad_input", "123")]:
    result = run_scenario(scenario, user)
    print(f"--- {scenario}")
    for line in result["trace"]:
        print("   ", line.split(" ", 1)[1])         # hide the cid to keep lines short
    print(f"    attempts={result['attempts']}  fallback={result['fallback']}")
    print(f"    customer sees: {result['answer']}")

### Which safety net saved each drill?

| Drill | What happened | Safety net |
| --- | --- | --- |
| healthy | worked first time | none needed |
| flaky | one temporary error | retry with backoff |
| down | every call failed | tries ran out, so **fallback** |
| slow | first call too slow | **timeout**, then retry |
| bad_input | wrong id | **escape path**, no retries |

**Check:** none of the runs hung, looped forever, or invented an answer.

---
### Your turn: predict first, then run (this is also today's assignment)
1. Add a new drill to `FAULTS`, for example `"mixed": ["slow", "error"]` (slow first, then a dropped line).
2. **Predict** the ending: done, fallback, or escape?
3. Run it and write one sentence saying which safety net saved the run.

In [ ]:
FAULTS["mixed"] = ["slow", "error"]
result = run_scenario("mixed")
for line in result["trace"]:
    print("   ", line.split(" ", 1)[1])
print("customer sees:", result["answer"])

**Answer:** both tries fail (a timeout, then a dropped line), so the tries run out and the **fallback** answers.

Try a second one of your own, then prove it cannot loop: its trace must end in a normal answer, `FALLBACK` or `escape path`.

---

## Concept Corner: the same ideas in other places
*Reading only. Nothing here changes today's lab; it just shows these ideas are shared across the industry.*

* **In a managed cloud:** services publish *quotas* and answer "too many requests" when you go over. Their client tools retry with *exponential backoff and jitter* and let you set *timeouts*. Bigger systems add a *circuit breaker* (stop calling a broken service for a while) and a *dead-letter queue* (park work that keeps failing for a person to look at), which is the cousin of our escape path.
* **In Claude's world:** the Claude API returns a *rate-limit error* when you send too much and an *overloaded error* when it is busy. Both are temporary, so retry with a wait. A malformed request will fail every time, so never retry it. The official client libraries already retry some errors for you, and the Agent SDK lets you cap how many turns an agent takes (our bounded loop).

| Today's idea | Managed / Claude name |
| --- | --- |
| Timeout | client timeout |
| Backoff + jitter | exponential backoff with jitter |
| Rate limit | quotas and throttling |
| Backpressure | load shedding, "overloaded" or 503 |
| Escalate | dead-letter queue, human review |
| Fallback | circuit breaker |
| Loop cap | maximum turns |

---

## Remember (the whole day in five lines)
1. Put a **timeout** on every outside call.
2. **Retry** temporary errors a few times with backoff and jitter.
3. Every failure ends in a **fallback** or a **human handoff**, never a hang.
4. **Limit** the load: wait (rate limit) or turn away (backpressure).
5. **Rehearse** failures with fire drills.

**Where this goes:** in Day 19, `agent.py::run_tool` keeps the same `MCP_ATTEMPTS = 2` and the same fallback sentence. **Tomorrow (Day 6):** the pretend card company is replaced by a real MCP server.